In [1]:
# =========================================================
# CISB5123 Lab 8 - Text Clustering
# Exercise 1 + Exercise 2
# =========================================================

import re
import numpy as np
import pandas as pd
from collections import Counter
from tabulate import tabulate

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

from gensim.models import Word2Vec
from nltk.stem import PorterStemmer

# -----------------------------
# TEXT PREPROCESSING FUNCTION
# -----------------------------
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)      # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)            # keep letters only
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens
              if word not in ENGLISH_STOP_WORDS and len(word) > 2]
    return " ".join(tokens)

# -----------------------------
# PURITY FUNCTION
# (same style as in lab sheet)
# -----------------------------
def calculate_purity(y_pred):
    total_samples = len(y_pred)
    cluster_label_counts = [Counter(y_pred)]
    purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
    return purity

# -----------------------------
# TF-IDF CLUSTERING FUNCTION
# -----------------------------
def run_tfidf_clustering(dataset, k=2, title="TF-IDF"):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(dataset)

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    y_pred = km.predict(X)

    print(f"\n{'='*60}")
    print(f"{title} CLUSTERING RESULTS")
    print(f"{'='*60}")

    table_data = [["Document", "Predicted Cluster"]]
    table_data.extend([[doc[:100] + ("..." if len(doc) > 100 else ""), cluster]
                       for doc, cluster in zip(dataset, y_pred)])
    print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

    print("\nTop terms per cluster:")
    order_centroids = km.cluster_centers_.argsort()[:, ::-1]
    terms = vectorizer.get_feature_names_out()

    for i in range(k):
        print(f"Cluster {i}:")
        for ind in order_centroids[i, :10]:
            print(" ", terms[ind])
        print()

    purity = calculate_purity(y_pred)
    print("Purity:", purity)

    return y_pred, purity

# -----------------------------
# WORD2VEC CLUSTERING FUNCTION
# -----------------------------
def run_word2vec_clustering(dataset, k=2, title="Word2Vec"):
    tokenized_dataset = [doc.split() for doc in dataset]

    word2vec_model = Word2Vec(
        sentences=tokenized_dataset,
        vector_size=100,
        window=5,
        min_count=1,
        workers=4,
        seed=42
    )

    X = np.array([
        np.mean([word2vec_model.wv[word] for word in doc if word in word2vec_model.wv], axis=0)
        for doc in tokenized_dataset
    ])

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    y_pred = km.predict(X)

    print(f"\n{'='*60}")
    print(f"{title} CLUSTERING RESULTS")
    print(f"{'='*60}")

    table_data = [["Document", "Predicted Cluster"]]
    table_data.extend([[doc[:100] + ("..." if len(doc) > 100 else ""), cluster]
                       for doc, cluster in zip(dataset, y_pred)])
    print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

    purity = calculate_purity(y_pred)
    print("Purity:", purity)

    return y_pred, purity

# =========================================================
# EXERCISE 1
# Compare before vs after preprocessing
# =========================================================

dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

processed_dataset = [preprocess_text(doc) for doc in dataset]

print("Original Dataset:")
for doc in dataset:
    print("-", doc)

print("\nPreprocessed Dataset:")
for doc in processed_dataset:
    print("-", doc)

# TF-IDF without preprocessing
_, tfidf_purity_before = run_tfidf_clustering(dataset, k=2, title="TF-IDF WITHOUT PREPROCESSING")

# TF-IDF with preprocessing
_, tfidf_purity_after = run_tfidf_clustering(processed_dataset, k=2, title="TF-IDF WITH PREPROCESSING")

# Word2Vec without preprocessing
_, w2v_purity_before = run_word2vec_clustering(dataset, k=2, title="Word2Vec WITHOUT PREPROCESSING")

# Word2Vec with preprocessing
_, w2v_purity_after = run_word2vec_clustering(processed_dataset, k=2, title="Word2Vec WITH PREPROCESSING")

print("\n" + "="*60)
print("EXERCISE 1 SUMMARY")
print("="*60)
print("TF-IDF Purity before preprocessing :", tfidf_purity_before)
print("TF-IDF Purity after preprocessing  :", tfidf_purity_after)
print("Word2Vec Purity before preprocessing :", w2v_purity_before)
print("Word2Vec Purity after preprocessing  :", w2v_purity_after)

# =========================================================
# EXERCISE 2
# Perform clustering on customer_complaints_1.csv
# =========================================================

print("\n" + "="*60)
print("EXERCISE 2 - CUSTOMER COMPLAINTS DATASET")
print("="*60)

# Load CSV
df = pd.read_csv("customer_complaints_1.csv")

print("\nColumns in dataset:", df.columns.tolist())
print("Dataset shape:", df.shape)

# The CSV uses lowercase 'text'
complaints = df["text"].dropna().astype(str).tolist()
processed_complaints = [preprocess_text(doc) for doc in complaints]

print("\nFirst 5 preprocessed complaints:")
for i, doc in enumerate(processed_complaints[:5], 1):
    print(f"{i}. {doc}")

# You can change k if your lecturer wants a specific number
k = 3

# TF-IDF on complaints
_, complaints_tfidf_purity = run_tfidf_clustering(
    processed_complaints,
    k=k,
    title=f"CUSTOMER COMPLAINTS TF-IDF (k={k})"
)

# Word2Vec on complaints
_, complaints_w2v_purity = run_word2vec_clustering(
    processed_complaints,
    k=k,
    title=f"CUSTOMER COMPLAINTS Word2Vec (k={k})"
)

print("\n" + "="*60)
print("EXERCISE 2 SUMMARY")
print("="*60)
print(f"Customer Complaints TF-IDF Purity   : {complaints_tfidf_purity}")
print(f"Customer Complaints Word2Vec Purity : {complaints_w2v_purity}")

Original Dataset:
- I love playing football on the weekends
- I enjoy hiking and camping in the mountains
- I like to read books and watch movies
- I prefer playing video games over sports
- I love listening to music and going to concerts

Preprocessed Dataset:
- love play footbal weekend
- enjoy hike camp mountain
- like read book watch movi
- prefer play video game sport
- love listen music go concert

TF-IDF WITHOUT PREPROCESSING CLUSTERING RESULTS
+-------------------------------------------------+---------------------+
| Document                                        |   Predicted Cluster |
+=================================================+=====================+
| I love playing football on the weekends         |                   1 |
+-------------------------------------------------+---------------------+
| I enjoy hiking and camping in the mountains     |                   1 |
+-------------------------------------------------+---------------------+
| I like to read books and